# Part 5: Machine Learning - Clustering Models

## Objectives
1. Select features for clustering
2. Train K-means, GMM, and HMM models
3. Find optimal number of clusters
4. Compare model performance
5. Identify market regimes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.data_loader import load_csv
from src.models import (
    standardize_features,
    find_optimal_k,
    train_kmeans,
    train_gmm,
    evaluate_clustering,
    assign_clusters_to_df
)
from src.visualization import plot_elbow_curve, plot_cluster_sizes

df = load_csv('../data/processed/merged_final_engineered.csv')
df['DATE'] = pd.to_datetime(df['DATE'])

## Feature Selection & Standardization

In [ ]:
# Select features for clustering
feature_cols = ['VIX_log', 'GPR_INDEX_log', 'REAL_YIELD', 'USD_RET', 'SP500_RET']
X = df[feature_cols].copy()

# Remove rows with missing values
X = X.dropna()
print(f"Shape after removing NaNs: {X.shape}")

# Standardize features
X_scaled, scaler = standardize_features(X)

## Find Optimal K

In [ ]:
# Test k from 2 to 10
k_range = range(2, 11)
optimal_k, scores = find_optimal_k(X_scaled, k_range, metric='silhouette')

# Plot elbow curve
plot_elbow_curve(list(k_range), scores, metric='Silhouette Score')

## Train K-Means (Primary Model)

In [ ]:
# Train K-means with optimal k
kmeans_labels, kmeans_model = train_kmeans(X_scaled, n_clusters=4)

# Evaluate
kmeans_eval = evaluate_clustering(kmeans_labels, X_scaled)

# Plot cluster sizes
plot_cluster_sizes(kmeans_labels)

## Train Gaussian Mixture Model

In [ ]:
# Train GMM
gmm_labels, gmm_model = train_gmm(X_scaled, n_components=4)

# Evaluate
gmm_eval = evaluate_clustering(gmm_labels, X_scaled)

## Assign Regimes to Dataset

In [ ]:
# Create a dataframe with clusters (align indices)
df_clustered = df[feature_cols].dropna().copy()
df_clustered['Regime'] = kmeans_labels

# Regime names
regime_names = {
    0: 'Calm / Bull Market',
    1: 'Financial Instability',
    2: 'Crisis / Panic',
    3: 'Inflationary Tightening'
}

df_clustered['Regime_Name'] = df_clustered['Regime'].map(regime_names)

print("\nRegime Distribution:")
print(df_clustered['Regime_Name'].value_counts())

# Save results
df_clustered.to_csv('../results/clustered_data.csv', index=False)
print("\n✅ Clustering complete!")